# Extract FIDE Calculation Pages for Additional Sample: `player_sample_1000_2`

**Purpose:**  
This notebook extracts FIDE rating-calculation page records for another additional 1,000-player sample (`player_sample_1000_2`). It documents the extended enrichment-data collection workflow beyond the frozen final 1,000-player sample used in the submitted report.


**Main outputs:**  
- FIDE calculation-page extraction records for `player_sample_1000_2`  
- Cleaned opponent-game records for this sample  
- Intermediate and final extraction files for possible future expansion


In [1]:
# ============================================================
# Notebook overview
# Purpose:
# - Extract FIDE rating-calculation data for player_sample_1000_2.
# - Support future expansion beyond the frozen final report sample.
# ============================================================
# From 701-800

In [2]:
# ============================================================
# Step 1: Import required libraries
# Purpose:
# - Load packages for file handling, web/browser extraction,
#   table parsing, data cleaning, and progress tracking.
# ============================================================

import pandas as pd
import numpy as np
import re
import time
import math
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO
from playwright.async_api import async_playwright

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", 80)

## 1. Setup and path configuration

This section loads required libraries and defines the input/output paths for extracting calculation-page data for this additional sample.


In [4]:
# ============================================================
# Step 2: Define input and output paths
# Purpose:
# - Set the input path for the player_sample_1000_2 file.
# - Set folders for raw, intermediate, and final extraction outputs.
# ============================================================

#sample_path = Path.home() / "Downloads" / "fide_history" / "indian_youth_1000_balanced_sample.csv"
# Or use representative sample:
# sample_path = Path.home() / "Downloads" / "fide_history" / "indian_youth_1000_player_features.csv"
player_sample_1000_output_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "player_sample_1000_2.csv"

player_sample = pd.read_csv(player_sample_1000_output_path)

player_sample.columns = player_sample.columns.str.strip()
player_sample["ID Number"] = player_sample["ID Number"].astype(str).str.strip()

player_ids = player_sample["ID Number"].dropna().drop_duplicates().tolist()

print("Players:", len(player_ids))
print(player_ids[:10])

Players: 100
['25973835', '429039118', '429005191', '25682652', '537023896', '48786047', '25785974', '429012635', '25773526', '429072832']


In [7]:
# ============================================================
# Step 3: Create output folders
# Purpose:
# - Ensure output directories exist before extraction files are written.
# ============================================================

rating_type = 0  # 0 = Standard, 1 = Rapid, 2 = Blitz

start_period = "2024-05-01"
end_period = "2025-04-01"

periods = pd.date_range(start_period, end_period, freq="MS")

print("Months:", len(periods))
print([p.strftime("%Y-%m-%d") for p in periods])

Months: 12
['2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01']


In [6]:
# ============================================================
# Step 4: Load additional player sample
# Purpose:
# - Read player_sample_1000_2 or the corresponding sample file.
# - Confirm the notebook is working on the intended additional sample.
# ============================================================

base_output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "fide_calculation_pages_2"

raw_html_dir = base_output_dir / "raw_html_2"
clean_dir = base_output_dir / "clean_player_months_2"
log_dir = base_output_dir / "logs_2"

raw_html_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

print(base_output_dir)

/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess/data/raw/fide_history/fide_calculation_pages_2


## 2. Load `player_sample_1000_2`

This section loads the additional 1,000-player sample used in this extraction run. This is separate from the fixed sample used in the final report.


In [11]:
# ============================================================
# Step 5: Inspect sample fields and row count
# Purpose:
# - Verify player count, key columns, and sample records before extraction.
# ============================================================

def extract_event_metadata_from_text(text_lines):
    events = []

    for i, line in enumerate(text_lines):
        if line == "Rc":
            if i >= 4:
                event_name = text_lines[i - 4]
                location_fed = text_lines[i - 3]
                start_date = text_lines[i - 2]
                end_date = text_lines[i - 1]

                if (
                    re.match(r"^\d{4}-\d{2}-\d{2}$", start_date)
                    and re.match(r"^\d{4}-\d{2}-\d{2}$", end_date)
                ):
                    events.append({
                        "tournament_name": event_name,
                        "event_location_fed": location_fed,
                        "event_start_date": start_date,
                        "event_end_date": end_date
                    })

    return events


def clean_fide_calculation_table(table, event_meta, fide_id, period, rating_type=0):
    df = table.copy()

    if df.shape[1] < 10 or df.shape[0] < 4:
        return pd.DataFrame()

    df = df.iloc[:, :10].copy()

    summary = df.iloc[1]

    event_rc = pd.to_numeric(summary.iloc[0], errors="coerce")
    event_ro = pd.to_numeric(summary.iloc[1], errors="coerce")
    event_score = pd.to_numeric(summary.iloc[5], errors="coerce")
    event_games = pd.to_numeric(summary.iloc[6], errors="coerce")
    event_chg = pd.to_numeric(summary.iloc[7], errors="coerce")
    event_k = pd.to_numeric(summary.iloc[8], errors="coerce")
    event_kchg = pd.to_numeric(summary.iloc[9], errors="coerce")

    opponent_df = df.iloc[3:].copy()

    opponent_df.columns = [
        "opponent_name",
        "opponent_title_1",
        "opponent_title_2",
        "opponent_rating_raw",
        "opponent_fed",
        "score",
        "n",
        "chg",
        "k",
        "k_chg"
    ]

    opponent_df = opponent_df[opponent_df["opponent_name"].notna()].copy()

    opponent_df = opponent_df[
        ~opponent_df["opponent_name"].astype(str).str.contains(
            "Rating difference", case=False, na=False
        )
    ].copy()

    opponent_df["opponent_fed"] = opponent_df["opponent_fed"].astype(str).str.strip()

    opponent_df = opponent_df[
        opponent_df["opponent_fed"].str.match(r"^[A-Z]{3}$", na=False)
    ].copy()

    if opponent_df.empty:
        return pd.DataFrame()

    opponent_df["opponent_title"] = (
        opponent_df[["opponent_title_1", "opponent_title_2"]]
        .astype(str)
        .replace("nan", "", regex=False)
        .agg(" ".join, axis=1)
        .str.strip()
    )

    opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)

    opponent_df["rating_difference_over_400_flag"] = (
        opponent_df["opponent_rating_raw"]
        .astype(str)
        .str.contains(r"\*", regex=True, na=False)
    )

    opponent_df["opponent_rating"] = (
        opponent_df["opponent_rating_raw"]
        .astype(str)
        .str.replace("*", "", regex=False)
        .str.strip()
    )

    opponent_df["opponent_rating"] = pd.to_numeric(
        opponent_df["opponent_rating"],
        errors="coerce"
    )

    for col in ["score", "n", "chg", "k", "k_chg"]:
        opponent_df[col] = pd.to_numeric(opponent_df[col], errors="coerce")

    opponent_df["fide_id"] = str(fide_id)
    opponent_df["period"] = period
    opponent_df["rating_type"] = rating_type

    opponent_df["tournament_name"] = event_meta.get("tournament_name")
    opponent_df["event_location_fed"] = event_meta.get("event_location_fed")
    opponent_df["event_start_date"] = event_meta.get("event_start_date")
    opponent_df["event_end_date"] = event_meta.get("event_end_date")

    opponent_df["event_rc"] = event_rc
    opponent_df["event_ro"] = event_ro
    opponent_df["event_score"] = event_score
    opponent_df["event_games"] = event_games
    opponent_df["event_chg"] = event_chg
    opponent_df["event_k"] = event_k
    opponent_df["event_k_chg"] = event_kchg

    keep_cols = [
        "fide_id", "period", "rating_type",
        "tournament_name", "event_location_fed",
        "event_start_date", "event_end_date",
        "event_rc", "event_ro", "event_score", "event_games",
        "event_chg", "event_k", "event_k_chg",
        "opponent_name", "opponent_title", "opponent_rating",
        "rating_difference_over_400_flag",
        "opponent_fed", "score", "n", "chg", "k", "k_chg"
    ]

    return opponent_df[keep_cols].reset_index(drop=True)

In [13]:
# ============================================================
# Step 6: Standardize FIDE IDs
# Purpose:
# - Convert FIDE IDs to a consistent format.
# - Prevent URL or join errors caused by mixed data types.
# ============================================================

async def extract_player_month(page, fide_id, period_str, rating_type=0, save_html=False):
    url = (
        "https://ratings.fide.com/calculations.phtml"
        f"?id_number={fide_id}&period={period_str}&rating={rating_type}"
    )

    await page.goto(url, wait_until="networkidle", timeout=60000)
    await page.wait_for_timeout(2000)

    html = await page.content()

    if save_html:
        player_html_dir = raw_html_dir / str(fide_id)
        player_html_dir.mkdir(parents=True, exist_ok=True)

        html_path = player_html_dir / f"{fide_id}_{period_str}_standard.html"
        html_path.write_text(html, encoding="utf-8")

    soup = BeautifulSoup(html, "html.parser")

    text_lines = [
        line.strip()
        for line in soup.get_text("\n", strip=True).split("\n")
        if line.strip()
    ]

    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        tables = []

    events = extract_event_metadata_from_text(text_lines)

    clean_tables = []

    for i, table in enumerate(tables):
        event_meta = events[i] if i < len(events) else {}

        clean_one = clean_fide_calculation_table(
            table=table,
            event_meta=event_meta,
            fide_id=fide_id,
            period=period_str,
            rating_type=rating_type
        )

        if not clean_one.empty:
            clean_tables.append(clean_one)

    if clean_tables:
        clean_df = pd.concat(clean_tables, ignore_index=True)
    else:
        clean_df = pd.DataFrame()

    status = {
        "fide_id": str(fide_id),
        "period": period_str,
        "rating_type": rating_type,
        "url": url,
        "tables_found": len(tables),
        "events_found": len(events),
        "clean_rows": clean_df.shape[0],
        "status": "data_found" if clean_df.shape[0] > 0 else "no_data_or_no_clean_rows"
    }

    return clean_df, status

In [15]:
# ============================================================
# Step 7: Define extraction months / rating periods
# Purpose:
# - Specify which FIDE calculation periods should be checked for each player.
# ============================================================

async def run_extraction_for_players(
    player_ids_to_run,
    periods,
    rating_type=0,
    save_html=False,
    delay_ms=1500
):
    all_status = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for idx, fide_id in enumerate(player_ids_to_run, start=1):
            print(f"\n===== Player {idx}/{len(player_ids_to_run)}: {fide_id} =====")

            player_clean_parts = []

            for period in periods:
                period_str = period.strftime("%Y-%m-%d")

                output_file = clean_dir / f"{fide_id}_{period_str}_standard_clean.csv"

                # Resume support: skip if already processed
                if output_file.exists():
                    print("Skipping existing:", output_file.name)

                    try:
                        existing_df = pd.read_csv(output_file)
                        clean_rows = existing_df.shape[0]
                    except Exception:
                        clean_rows = None

                    all_status.append({
                        "fide_id": str(fide_id),
                        "period": period_str,
                        "rating_type": rating_type,
                        "status": "skipped_existing",
                        "clean_rows": clean_rows
                    })
                    continue

                print("Processing:", fide_id, period_str)

                try:
                    clean_df, status = await extract_player_month(
                        page=page,
                        fide_id=fide_id,
                        period_str=period_str,
                        rating_type=rating_type,
                        save_html=save_html
                    )

                    # Save even if empty, so resume knows it was processed
                    clean_df.to_csv(output_file, index=False)

                    all_status.append(status)

                    if not clean_df.empty:
                        player_clean_parts.append(clean_df)
                        print("Rows:", clean_df.shape[0])
                    else:
                        print("No rows.")

                except Exception as e:
                    print("Failed:", fide_id, period_str, repr(e))

                    all_status.append({
                        "fide_id": str(fide_id),
                        "period": period_str,
                        "rating_type": rating_type,
                        "status": "failed",
                        "clean_rows": 0,
                        "error": repr(e)
                    })

                await page.wait_for_timeout(delay_ms)

            # Save player-level combined file
            if player_clean_parts:
                player_all = pd.concat(player_clean_parts, ignore_index=True)
                player_output = clean_dir / f"{fide_id}_all_months_standard_clean.csv"
                player_all.to_csv(player_output, index=False)
                print("Saved player file:", player_output, "Rows:", player_all.shape[0])

            # Save status after every player
            status_df = pd.DataFrame(all_status)
            status_path = log_dir / "fide_calculation_extraction_status.csv"
            status_df.to_csv(status_path, index=False)

        await browser.close()

    return pd.DataFrame(all_status)

## 3. Prepare FIDE IDs and extraction periods

This section standardizes player identifiers and defines the rating months/periods for which FIDE calculation pages are checked.


In [17]:
# ============================================================
# Step 8: Build FIDE calculation-page URL helper
# Purpose:
# - Create or test the URL pattern used to access FIDE calculation pages.
# ============================================================

status_all = await run_extraction_for_players(
    player_ids_to_run=player_ids,
    periods=periods,
    rating_type=0,
    save_html=False,
    delay_ms=2000
)

display(status_all)


===== Player 1/100: 25973835 =====
Processing: 25973835 2024-05-01
No rows.
Processing: 25973835 2024-06-01


/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_68823/1362845381.py:84: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)


Rows: 4
Processing: 25973835 2024-07-01


/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_68823/1362845381.py:84: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)


Rows: 5
Processing: 25973835 2024-08-01
No rows.
Processing: 25973835 2024-09-01
No rows.
Processing: 25973835 2024-10-01
No rows.
Processing: 25973835 2024-11-01
No rows.
Processing: 25973835 2024-12-01
No rows.
Processing: 25973835 2025-01-01
No rows.
Processing: 25973835 2025-02-01


...

/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_68823/1362845381.py:84: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)


Rows: 8
Processing: 33365318 2025-02-01
No rows.
Processing: 33365318 2025-03-01


/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_68823/1362845381.py:84: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)


Rows: 2
Processing: 33365318 2025-04-01
No rows.
Saved player file: /Users/arunkumar/Downloads/fide_history/fide_calculation_pages_2/clean_player_months_2/33365318_all_months_standard_clean.csv Rows: 33


,fide_id,period,rating_type,url,tables_found,events_found,clean_rows,status,error
0,25973835,2024-05-01,0,https://ratings.fide.com/calculations.phtml?id_number=25973835&period=2024-0...,0.0,0.0,0,no_data_or_no_clean_rows,NaN
1,25973835,2024-06-01,0,https://ratings.fide.com/calculations.phtml?id_number=25973835&period=2024-0...,1.0,1.0,4,data_found,NaN
2,25973835,2024-07-01,0,https://ratings.fide.com/calculations.phtml?id_number=25973835&period=2024-0...,1.0,1.0,5,data_found,NaN
3,25973835,2024-08-01,0,https://ratings.fide.com/calculations.phtml?id_number=25973835&period=2024-0...,0.0,0.0,0,no_data_or_no_clean_rows,NaN
4,25973835,2024-09-01,0,https://ratings.fide.com/calculations.phtml?id_number=25973835&period=2024-0...,0.0,0.0,0,no_data_or_no_clean_rows,NaN
...,...,...,...,...,...,...,...,...,...
1195,33365318,2024-12-01,0,https://ratings.fide.com/calculations.phtml?id_number=33365318&period=2024-1...,0.0,0.0,0,no_data_or_no_clean_rows,NaN
1196,33365318,2025-01-01,0,https://ratings.fide.com/calculations.phtml?id_number=33365318&period=2025-0...,1.0,1.0,8,data_found,NaN
1197,33365318,2025-02-01,0,https://ratings.fide.com/calculations.phtml?id_number=33365318&period=2025-0...,0.0,0.0,0,no_data_or_no_clean_rows,NaN
1198,33365318,2025-03-01,0,https://ratings.fide.com/calculations.phtml?id_number=33365318&period=2025-0...,1.0,1.0,2,data_found,NaN
